In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier


# ============================================================
# EXPERIMENT 31
# Fast Identity Encoding + Interaction Attack
#
# Goal:
#   Start from the strongest known representation:
#       identity target + identity frequency
#
#   Then test one targeted extension:
#       selected pair + triple identity target encodings
#
# No submission generation.
# No prediction CSV.
# Everything stays in memory.
# ============================================================


PROJECT_ROOT = r"C:\Users\aakif\Documents\DataCompetition"
TRAIN_PATH = PROJECT_ROOT + r"\data\train.csv"

TARGET = "Will_Buy_EV"
ID_COL = "id"

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("")
print("Numeric columns:")
print(numeric_cols)


# ============================================================
# IDENTITY ENCODING
# ============================================================

def make_identity_key(series):
    return series.astype("string").fillna("__MISSING__")


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        "value": values,
        "target": target.to_numpy()
    })

    global_mean = float(target.mean())

    stats = (
        temp.groupby("value", dropna=False)["target"]
        .agg(["mean", "count"])
    )

    smoothed = (
        stats["count"] * stats["mean"]
        + smoothing * global_mean
    ) / (stats["count"] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return (
        values.map(mapping)
        .fillna(global_mean)
        .astype(float)
    )


def make_key(frame, columns):
    key = make_identity_key(frame[columns[0]])

    for col in columns[1:]:
        key = key + "||" + make_identity_key(frame[col])

    return key


def add_identity_target_features(
    X_fit,
    y_fit,
    X_apply,
    columns,
    n_splits=3,
    smoothing=20,
    add_frequency=True
):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:

        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf.split(X_fit, y_fit):

            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = (
                apply_mapping(
                    fit_keys.iloc[fold_idx],
                    mapping,
                    global_mean
                ).to_numpy()
            )

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[f"{col}__identity_target"] = oof_values

        X_apply[f"{col}__identity_target"] = (
            apply_mapping(
                apply_keys,
                full_mapping,
                full_global_mean
            ).to_numpy()
        )

        if add_frequency:

            frequencies = fit_keys.value_counts(dropna=False)

            X_fit[f"{col}__identity_frequency"] = (
                fit_keys.map(frequencies)
                .fillna(0)
                .astype(float)
                .to_numpy()
            )

            X_apply[f"{col}__identity_frequency"] = (
                apply_keys.map(frequencies)
                .fillna(0)
                .astype(float)
                .to_numpy()
            )

    return X_fit, X_apply


def add_group_identity_target(
    X_fit,
    y_fit,
    X_apply,
    groups,
    n_splits=3,
    smoothing=20
):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for columns in groups:

        feature_name = (
            "__x__".join(columns)
            + "__group_identity_target"
        )

        fit_keys = make_key(X_fit, columns)
        apply_keys = make_key(X_apply, columns)

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf.split(X_fit, y_fit):

            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = (
                apply_mapping(
                    fit_keys.iloc[fold_idx],
                    mapping,
                    global_mean
                ).to_numpy()
            )

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[feature_name] = oof_values

        X_apply[feature_name] = (
            apply_mapping(
                apply_keys,
                full_mapping,
                full_global_mean
            ).to_numpy()
        )

    return X_fit, X_apply


# ============================================================
# XGBOOST
# ============================================================

def build_preprocessor(X_frame):

    numeric = X_frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical = X_frame.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric),
        ("cat", categorical_pipeline, categorical)
    ])


def run_xgb(X_tr, y_tr, X_va, y_va, label):

    print("")
    print("------------------------------------------------------------")
    print(label)
    print("------------------------------------------------------------")

    print("Building preprocessor...")

    preprocessor = build_preprocessor(X_tr)

    X_tr_encoded = preprocessor.fit_transform(X_tr)
    X_va_encoded = preprocessor.transform(X_va)

    print(
        "Encoded shape:",
        X_tr_encoded.shape
    )

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr_encoded,
        y_tr,
        eval_set=[(X_va_encoded, y_va)],
        verbose=False
    )

    predictions = model.predict_proba(
        X_va_encoded
    )[:, 1]

    score = roc_auc_score(
        y_va,
        predictions
    )

    print("")
    print(f"{label} ROC-AUC: {score:.6f}")

    return score


# ============================================================
# BASELINE
# Proven identity target + frequency representation
# ============================================================

print("")
print("============================================================")
print("31A - PROVEN IDENTITY + FREQUENCY BASELINE")
print("============================================================")

print("")
print("Building leakage-safe identity target features...")

X_base_train, X_base_valid = add_identity_target_features(
    X_train,
    y_train,
    X_valid,
    numeric_cols,
    n_splits=3,
    smoothing=20,
    add_frequency=True
)

print(
    "Baseline feature count:",
    X_base_train.shape[1]
)

score_base = run_xgb(
    X_base_train,
    y_train,
    X_base_valid,
    y_valid,
    "31A_Identity_Target_Frequency"
)

print("")
print("Known Experiment 23B benchmark: 0.945243")
print(f"31A result: {score_base:.6f}")
print(
    f"Difference vs 23B: "
    f"{score_base - 0.945243:+.6f}"
)


# ============================================================
# TARGETED EXTENSION
#
# Only continue to the interaction attack after the baseline
# has completed. This is deliberately limited to a small number
# of high-information groups rather than a giant sweep.
# ============================================================

print("")
print("============================================================")
print("31B - TARGETED PAIR + TRIPLE IDENTITY ATTACK")
print("============================================================")

pair_groups = [
    ("Age", "Annual_Income_USD"),
    ("Annual_Income_USD", "Daily_Commute_km"),
    ("Daily_Commute_km", "Charging_Stations_Near_Work"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
    ("Number_of_Cars_Owned", "Annual_Income_USD")
]

triple_groups = [
    (
        "Age",
        "Annual_Income_USD",
        "Daily_Commute_km"
    ),
    (
        "Annual_Income_USD",
        "Daily_Commute_km",
        "Charging_Stations_Near_Work"
    )
]

print("")
print("Adding pair identity target encodings...")

X_attack_train, X_attack_valid = add_group_identity_target(
    X_base_train,
    y_train,
    X_base_valid,
    pair_groups,
    n_splits=3,
    smoothing=20
)

print(
    "Feature count after pairs:",
    X_attack_train.shape[1]
)

print("")
print("Adding two targeted triple identity encodings...")

X_attack_train, X_attack_valid = add_group_identity_target(
    X_attack_train,
    y_train,
    X_attack_valid,
    triple_groups,
    n_splits=3,
    smoothing=20
)

print(
    "Final feature count:",
    X_attack_train.shape[1]
)

score_attack = run_xgb(
    X_attack_train,
    y_train,
    X_attack_valid,
    y_valid,
    "31B_Identity_Frequency_Pair_Triple"
)


# ============================================================
# RESULTS
# ============================================================

results = pd.DataFrame([
    {
        "Experiment": "31A_Identity_Target_Frequency",
        "ROC_AUC": score_base
    },
    {
        "Experiment": "31B_Identity_Frequency_Pair_Triple",
        "ROC_AUC": score_attack
    }
])

results = results.sort_values(
    "ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("")
print("")
print("============================================================")
print("EXPERIMENT 31 RESULTS")
print("============================================================")

print(
    results.to_string(index=False)
)

best_score = float(
    results.iloc[0]["ROC_AUC"]
)

print("")
print(f"Known 23B benchmark: {0.945243:.6f}")
print(f"Best Experiment 31: {best_score:.6f}")
print(
    f"Difference vs 23B: "
    f"{best_score - 0.945243:+.6f}"
)

if best_score >= 0.950000:
    print("")
    print(">>> 0.950+ GATE PASSED <<<")
    print("This representation is worth taking further.")

if best_score >= 0.960000:
    print("")
    print(">>> 0.960+ BREAKTHROUGH <<<")

print("")
print("No submission was generated.")
print("No prediction CSV was generated.")
print("Experiment 31 complete.")

Training rows: 534932
Validation rows: 133733
Numeric columns: 7
Categorical columns: 6

Numeric columns:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

31A - PROVEN IDENTITY + FREQUENCY BASELINE

Building leakage-safe identity target features...
Baseline feature count: 27

------------------------------------------------------------
31A_Identity_Target_Frequency
------------------------------------------------------------
Building preprocessor...
Encoded shape: (534932, 38)

31A_Identity_Target_Frequency ROC-AUC: 0.945243

Known Experiment 23B benchmark: 0.945243
31A result: 0.945243
Difference vs 23B: +0.000000

31B - TARGETED PAIR + TRIPLE IDENTITY ATTACK

Adding pair identity target encodings...
Feature count after pairs: 32

Adding two targeted triple identity encodings...
Final feature count: 34

------------------------------------------------------------
31B_I